### Merge first-order features with the target label
Combining the pre-outcome features (built from each customer's first order only) 
with the second-purchase label from the EDA step. Only the valid cohort — customers 
with enough observation time — gets kept.

In [1]:
import pandas as pd
import numpy as np

first_order_features = pd.read_csv("../data/processed/first_order_features.csv")
labeled_customers = pd.read_csv("../data/processed/customer_features_labeled.csv")

# Keep only the target-relevant columns from the labeled table —
# everything else there was computed across ALL orders and would leak
target_cols = labeled_customers[["customer_unique_id", "made_second_purchase"]]

# Inner join: only customers present in both (valid cohort AND has first-order features)
model_data = first_order_features.merge(target_cols, on="customer_unique_id", how="inner")

print(f"Merged dataset: {model_data.shape[0]:,} rows, {model_data.shape[1]} columns")
print(f"Target balance: {model_data['made_second_purchase'].mean():.2%} positive")
model_data.head()

Merged dataset: 55,907 rows, 15 columns
Target balance: 3.97% positive


,customer_unique_id,customer_state,first_purchase_date,n_items,n_distinct_products,items_total_price,total_freight,product_category_name,payment_total,max_installments,payment_type,delivery_days,delivery_delay_days,review_score,made_second_purchase
0,e8d40b1577995fcf06527c519ac56679,PR,2017-09-04 15:43:56,1,1,85.00,15.34,eletronicos,100.34,1.0,voucher,14.0,-9.0,5.0,0
1,f4e2749a803a0a9019e6c69bc0e1b2a5,MG,2018-01-19 11:05:41,1,1,13.99,14.10,perfumaria,28.09,1.0,credit_card,31.0,10.0,1.0,0
2,e8092d3af4962027cd1f45d669494cd6,RS,2017-05-06 12:28:42,1,1,798.00,56.89,pet_shop,854.89,6.0,credit_card,13.0,-14.0,5.0,0
3,36023a7f667578e554e3a308c9d17598,SC,2017-06-06 13:27:53,1,1,39.90,15.10,informatica_acessorios,55.00,2.0,credit_card,10.0,-13.0,5.0,0
4,86246b1ae68b9e35ca0d5b6dc0350333,RJ,2017-02-24 17:32:02,1,1,319.90,17.94,perfumaria,337.84,3.0,credit_card,11.0,-23.0,5.0,0


In [2]:
model_data.isnull().sum()

customer_unique_id          0
customer_state              0
first_purchase_date         0
n_items                     0
n_distinct_products         0
items_total_price           0
total_freight               0
product_category_name    1021
payment_total               1
max_installments            1
payment_type                1
delivery_days               2
delivery_delay_days         2
review_score              418
made_second_purchase        0
dtype: int64

### Check for missing values
Payment, review, and delivery fields can be null — reviews may not exist yet, and 
some orders may be missing delivery timestamps. Deciding how to handle each before 
modeling.

In [3]:
missing_summary = model_data.isnull().sum().sort_values(ascending=False)
missing_pct = (missing_summary / len(model_data) * 100).round(2)
pd.DataFrame({"missing_count": missing_summary, "missing_pct": missing_pct})

,missing_count,missing_pct
product_category_name,1021,1.83
review_score,418,0.75
delivery_days,2,0.00
delivery_delay_days,2,0.00
payment_total,1,0.00
max_installments,1,0.00
payment_type,1,0.00
total_freight,0,0.00
items_total_price,0,0.00
n_distinct_products,0,0.00


### Handling missing values
Each column's gap has a different cause, so each gets its own fix rather than 
one blanket rule. Category and review gaps are meaningful (no data left, not 
necessarily bad data) and get flagged/filled. The handful of rows missing 
payment or delivery info are just dropped too few to matter.


In [4]:
# Drop the tiny number of rows with missing payment/delivery info —
# not enough rows to justify imputation, and they may indicate
# a broken order record anyway
model_data = model_data.dropna(
    subset=["delivery_days", "delivery_delay_days", "payment_total",
            "max_installments", "payment_type"]
)

# Missing category likely means the product had no listed category in
# the source data — treat "unknown" as its own valid category rather
# than dropping potentially useful rows
model_data["product_category_name"] = model_data["product_category_name"].fillna("unknown")

# Missing review means no review was left — that's different from a bad
# review, so we flag it explicitly and fill the score with the median
# (keeps the average review score undistorted)
model_data["has_review"] = model_data["review_score"].notnull().astype(int)
model_data["review_score"] = model_data["review_score"].fillna(model_data["review_score"].median())

print(f"Remaining rows after cleanup: {len(model_data):,}")
print(f"Remaining nulls:\n{model_data.isnull().sum().sum()}")

Remaining rows after cleanup: 55,904
Remaining nulls:
0


In [18]:
model_data["review_score"].value_counts()

review_score
5.0    32532
4.0    11133
1.0     5556
3.0     4854
2.0     1780
4.5       21
2.5       13
3.5       12
1.5        3
Name: count, dtype: int64